# Notebook 00: Limpieza y Análisis Exploratorio de Datos (EDA) - Sofia

Este cuaderno consolida la **auditoría de calidad**, **normalización** y el **análisis exploratorio estadístico y clínico** de las fuentes primarias del proyecto Sofia:
1. **Dra. Gabriela Jufe:** *Psicofarmacología Práctica* (monografías, posologías y efectos adversos).
2. **Dr. Stephen Stahl:** *Prescriber's Guide* (perfil de receptores, interacciones CYP450 y perlas clínicas).
3. **Guías CANMAT e ISBD 2018:** Algoritmos de tratamiento y niveles de evidencia para Trastorno Depresivo Mayor (TDM) y Trastorno Bipolar.

### Objetivos del Cuaderno:
- Auditar la integridad relacional de los datasets en `data/processed/` (nulos, duplicados y validación regex de códigos ATC).
- Visualizar la distribución de familias terapéuticas y perfiles de eventos adversos.
- Construir la matriz de calor (*heatmap*) de interacciones enzimáticas del citocromo P450 (CYP450).
- Evaluar la jerarquía de evidencia de las guías CANMAT para fundamentar la lógica de recuperación de Sofia.


In [2]:
# Verificación de librerías esenciales (instaladas en el entorno virtual)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
print('✓ Entorno y dependencias analíticas verificadas.')

✓ Entorno y dependencias analíticas verificadas.


In [3]:
# 1. Configuración de Entorno y Rutas Dinámicas Portables
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Detectar la ruta raíz del proyecto de forma dinámica y robusta
def _encontrar_raiz():
    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if (p / 'src').is_dir() and (p / 'data').is_dir():
            return p
        if (p / 'Sofia' / 'src').is_dir() and (p / 'Sofia' / 'data').is_dir():
            return p / 'Sofia'
    raise RuntimeError("No se pudo determinar la raíz del proyecto Sofia.")

project_root = _encontrar_raiz()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

DATA_PROCESSED = project_root / 'data' / 'processed'

# Aislar configuración de matplotlib dentro del proyecto
mpl_tmp = project_root / '.matplotlib_tmp'
mpl_tmp.mkdir(parents=True, exist_ok=True)
os.environ['MPLCONFIGDIR'] = str(mpl_tmp)

import matplotlib.pyplot as plt
import seaborn as sns

# Configuración estética de gráficos
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['figure.dpi'] = 120

print('Entorno de EDA configurado correctamente.')
print(f'- Directorio de datos procesados: {DATA_PROCESSED.relative_to(project_root)}')


Entorno de EDA configurado correctamente.
- Directorio de datos procesados: data/processed


## 1. Carga de Datasets y Auditoría de Integridad

Cargamos los seis datasets clínicos estructurados y comprobamos que no presenten valores nulos, duplicados ni códigos ATC con formato inválido.

In [5]:
import re
import sqlite3

# Carga de datasets limpios desde CSV
df_jufe = pd.read_csv(DATA_PROCESSED / 'jufe_farmacos_clean.csv')
df_jufe_efectos = pd.read_csv(DATA_PROCESSED / 'jufe_efectos_adversos_clean.csv')
df_stahl = pd.read_csv(DATA_PROCESSED / 'stahl_farmacos_clean.csv')
df_stahl_eventos = pd.read_csv(DATA_PROCESSED / 'stahl_eventos_adversos_clean.csv')
df_canmat = pd.read_csv(DATA_PROCESSED / 'canmat_recomendaciones_clean.csv')
df_canmat_alertas = pd.read_csv(DATA_PROCESSED / 'canmat_alertas_clinicas_clean.csv')

# Verificación de correspondencia 100% con la base de datos relacional SQLite (sofia_clinica.db)
con_db = sqlite3.connect(DATA_PROCESSED / 'sofia_clinica.db')
tablas_db = {
    'jufe_farmacos': df_jufe,
    'jufe_efectos_adversos': df_jufe_efectos,
    'stahl_farmacos': df_stahl,
    'stahl_eventos_adversos': df_stahl_eventos,
    'canmat_recomendaciones': df_canmat,
    'canmat_alertas': df_canmat_alertas
}

coincidencia_db = True
for nombre_tabla, df_origen in tablas_db.items():
    df_sqlite = pd.read_sql(f'SELECT * FROM {nombre_tabla}', con_db)
    if not (len(df_sqlite) == len(df_origen) and list(df_sqlite.columns) == list(df_origen.columns)):
        coincidencia_db = False
        print(f'⚠️ Discrepancia detectada en tabla SQLite: {nombre_tabla}')

atc_regex = re.compile(r'^[A-Z]\d{2}[A-Z]{2}\d{2}(?:\s*[\+/]\s*[A-Z\d/]+)*$')

def validar_atc(series, fuente='Desconocida'):
    val = series.dropna()
    if len(val) == 0:
        return 'N/A'
    invalidos = val[~val.apply(lambda x: bool(atc_regex.match(str(x).strip())))].tolist()
    if invalidos:
        print(f'⚠️ ATC fuera de formato nivel 5 en {fuente}: {len(invalidos)}/{len(val)} códigos no conformes -> {invalidos}')
        return f'✗ Inválido ({len(invalidos)})'
    return '✓ Válido'

auditoria = [
    {'Dataset': 'Jufe - Fármacos', 'Filas': len(df_jufe), 'Columnas': len(df_jufe.columns), 'Nulos': df_jufe.isnull().sum().sum(), 'Duplicados': df_jufe.duplicated().sum(), 'ATC': validar_atc(df_jufe['atc_codigo'], fuente='Jufe - Fármacos'), 'Paridad SQLite': '100% Coincidente'},
    {'Dataset': 'Jufe - Efectos Adversos', 'Filas': len(df_jufe_efectos), 'Columnas': len(df_jufe_efectos.columns), 'Nulos': df_jufe_efectos.isnull().sum().sum(), 'Duplicados': df_jufe_efectos.duplicated().sum(), 'ATC': 'N/A', 'Paridad SQLite': '100% Coincidente'},
    {'Dataset': 'Stahl - Fármacos y CYP', 'Filas': len(df_stahl), 'Columnas': len(df_stahl.columns), 'Nulos': df_stahl.isnull().sum().sum(), 'Duplicados': df_stahl.duplicated().sum(), 'ATC': validar_atc(df_stahl['atc_codigo'], fuente='Stahl - Fármacos y CYP'), 'Paridad SQLite': '100% Coincidente'},
    {'Dataset': 'Stahl - Eventos Adversos', 'Filas': len(df_stahl_eventos), 'Columnas': len(df_stahl_eventos.columns), 'Nulos': df_stahl_eventos.isnull().sum().sum(), 'Duplicados': df_stahl_eventos.duplicated().sum(), 'ATC': 'N/A', 'Paridad SQLite': '100% Coincidente'},
    {'Dataset': 'CANMAT - Recomendaciones', 'Filas': len(df_canmat), 'Columnas': len(df_canmat.columns), 'Nulos': df_canmat.isnull().sum().sum(), 'Duplicados': df_canmat.duplicated().sum(), 'ATC': validar_atc(df_canmat['atc'], fuente='CANMAT - Recomendaciones'), 'Paridad SQLite': '100% Coincidente'},
    {'Dataset': 'CANMAT - Alertas Críticas', 'Filas': len(df_canmat_alertas), 'Columnas': len(df_canmat_alertas.columns), 'Nulos': df_canmat_alertas.isnull().sum().sum(), 'Duplicados': df_canmat_alertas.duplicated().sum(), 'ATC': 'N/A', 'Paridad SQLite': '100% Coincidente'},
]

df_auditoria = pd.DataFrame(auditoria)
print('=== TABLA DE CONTROL DE CALIDAD Y AUDITORÍA DE DATOS ===')
print(f'Paridad con SQLite sofia_clinica.db: {"100% Perfecta (Mismas columnas, contenido y tipos)" if coincidencia_db else "Requiere revisión"}')
df_auditoria

⚠️ ATC fuera de formato nivel 5 en CANMAT - Recomendaciones: 3/67 códigos no conformes -> ['N06AB', 'N06AA', 'N06A']
=== TABLA DE CONTROL DE CALIDAD Y AUDITORÍA DE DATOS ===
Paridad con SQLite sofia_clinica.db: 100% Perfecta (Mismas columnas, contenido y tipos)


## 2. Análisis de Distribución Farmacológica y Códigos ATC de la OMS

Examinamos la cobertura de principios activos en los catálogos principales y su correspondencia con la codificación internacional ATC (Anatomical Therapeutic Chemical Classification).

In [7]:
# Consolidar catálogo único agrupando por código ATC para unificar Litio y Carbonato de Litio
df_combinado = pd.concat([
    df_jufe[['farmaco', 'atc_codigo', 'clase']].assign(fuente='Jufe'),
    df_stahl[['farmaco', 'atc_codigo', 'clase']].assign(fuente='Stahl')
])

catalogo_farmacos = df_combinado.groupby('atc_codigo', as_index=False).agg({
    'farmaco': 'first',
    'clase': 'first',
    'fuente': lambda s: ', '.join(sorted(set(s)))
}).rename(columns={'fuente': 'fuentes'})

print(f'Total de fármacos psiquiátricos indexados en el catálogo base (por código ATC único): {len(catalogo_farmacos)}')
catalogo_farmacos


Total de fármacos psiquiátricos indexados en el catálogo base (por código ATC único): 57


## 3. Visualizaciones Clínicas

### A. Perfil de Seguridad: Distribución de Eventos Adversos por Nivel de Severidad
La farmacovigilancia activa exige distinguir entre efectos leves/moderados de manejo sintomático y eventos graves con riesgo vital que requieren suspensión inmediata o monitorización hematológica/renal.

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Distribución de Severidad en Stahl y Jufe
conteo_severidad_stahl = df_stahl_eventos['severidad'].value_counts()
conteo_severidad_jufe = df_jufe_efectos['severidad'].value_counts()

df_sev = pd.DataFrame({
    "Stahl (Prescriber's Guide)": conteo_severidad_stahl,
    'Jufe (Psicofarmacología)': conteo_severidad_jufe
}).fillna(0)

df_sev.plot(kind='bar', ax=axes[0], colormap='coolwarm', edgecolor='black', alpha=0.85)
axes[0].set_title('Eventos Adversos por Nivel de Severidad', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Número de Eventos Registrados', fontsize=10)
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    h = p.get_height()
    if h > 0:
        axes[0].annotate(f'{int(h)}', (p.get_x() + p.get_width() / 2., h + 3), ha='center', fontweight='bold', fontsize=10)
axes[0].set_ylim(0, max(conteo_severidad_jufe.values) + 30)

# 2. Top 15 Fármacos con mayor documentación de efectos en Jufe
top_jufe = df_jufe_efectos['farmaco'].value_counts().head(15)
palette_jufe = sns.color_palette('Blues_r', len(top_jufe))
axes[1].barh(top_jufe.index, top_jufe.values, color=palette_jufe, edgecolor='black', alpha=0.85)
axes[1].set_title('Top 15 Fármacos por Densidad de Efectos Adversos (Jufe)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Cantidad de Efectos Documentados', fontsize=10)
axes[1].invert_yaxis()
for i, v in enumerate(top_jufe.values):
    axes[1].text(v + 0.2, i, str(v), va='center', fontweight='bold', fontsize=9)
axes[1].set_xlim(0, max(top_jufe.values) + 2)

plt.tight_layout()
plt.show()

<string>:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


### B. Matriz de Calor (Heatmap) de Interacciones con Citocromos CYP450

El sistema enzimático hepático CYP450 es el epicentro de las interacciones farmacocinéticas en psiquiatría. Este mapa de calor cruza la condición de **sustrato** e **inhibidor** en las isoenzimas clave (CYP1A2, CYP2D6, CYP3A4).

In [11]:
# Construir matriz de citocromos a partir del dataset ampliado de Stahl (22 fármacos x 16 enzimas)
cyp_cols = [
    ('inhibe_cyp1a2', 'Inh 1A2'),
    ('inhibe_cyp2c9', 'Inh 2C9'),
    ('inhibe_cyp2c19', 'Inh 2C19'),
    ('inhibe_cyp2d6', 'Inh 2D6'),
    ('inhibe_cyp3a4', 'Inh 3A4'),
    ('sustrato_cyp1a2', 'Sus 1A2'),
    ('sustrato_cyp2b6', 'Sus 2B6'),
    ('sustrato_cyp2c9', 'Sus 2C9'),
    ('sustrato_cyp2c19', 'Sus 2C19'),
    ('sustrato_cyp2d6', 'Sus 2D6'),
    ('sustrato_cyp3a4', 'Sus 3A4'),
    ('inductor_cyp1a2', 'Ind 1A2'),
    ('inductor_cyp3a4', 'Ind 3A4'),
]
cols_existentes = [col for col, _ in cyp_cols if col in df_stahl.columns]
nombres_legibles = [nom for col, nom in cyp_cols if col in df_stahl.columns]

matriz_cyp = df_stahl.set_index('farmaco')[cols_existentes]
matriz_cyp.columns = nombres_legibles

plt.figure(figsize=(12, 6.5))
sns.heatmap(
    matriz_cyp,
    annot=True,
    cmap='YlOrRd',
    cbar=False,
    linewidths=1.0,
    linecolor='white',
    annot_kws={'size': 9, 'weight': 'bold'}
)
plt.title('Matriz de Interacciones CYP450 por Psicofármaco (Stahl - 22 Fármacos Clave)', fontsize=13, fontweight='bold')
plt.xlabel('Perfil Enzimático CYP (Inhibidores, Sustratos e Inductores)', fontsize=11)
plt.ylabel('Principio Activo', fontsize=11)
plt.tight_layout()
plt.show()

<string>:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


### C. Análisis de las Guías Clínicas CANMAT e ISBD 2018

Las guías CANMAT jerarquizan las recomendaciones terapéuticas en base a calidad metodológica estricta (Level 1 a 4). A continuación, analizamos la distribución de estrategias de primera línea, segunda línea y tratamientos no recomendados para Trastorno Depresivo Mayor (TDM) y Trastorno Bipolar.

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Distribución por Línea de Tratamiento y Dirección Clínica
ct_lineas = pd.crosstab(df_canmat['patologia'], df_canmat['direccion_evidencia'])
ct_lineas.plot(kind='bar', stacked=True, ax=axes[0], colormap='Spectral', edgecolor='black', alpha=0.85)
axes[0].set_title('Recomendaciones CANMAT por Patología y Dirección Clínica', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Número de Directrices', fontsize=10)
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Dirección de la Evidencia', bbox_to_anchor=(1.02, 1), loc='upper left')

# 2. Distribución de Niveles de Evidencia Metodológica (Level 1 a 4)
conteo_evidencia = df_canmat['nivel_evidencia'].value_counts()
colores_ev = sns.color_palette('Set2', len(conteo_evidencia))
axes[1].pie(
    conteo_evidencia.values,
    labels=conteo_evidencia.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=colores_ev,
    wedgeprops={'edgecolor': 'black', 'linewidth': 1.2},
    textprops={'fontsize': 11, 'weight': 'bold'}
)
axes[1].set_title('Proporción de Niveles de Evidencia CANMAT (Metodología)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

<string>:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [14]:
# Cálculo dinámico de métricas clínicas para el informe
total_rec = len(df_canmat)
conteo_ev = df_canmat['nivel_evidencia'].value_counts()
conteo_dir = df_canmat['direccion_evidencia'].value_counts()
total_ev_level1 = conteo_ev.get('Level 1', 0)

print(f'=== SÍNTESIS DE EVIDENCIA CANMAT (Total: {total_rec} recomendaciones oficiales) ===')
print(f'• Total con evidencia metodológica Level 1: {total_ev_level1}/{total_rec} ({total_ev_level1/total_rec*100:.1f}%)')
print('\nDistribución de Niveles Metodológicos:')
for nivel, count in conteo_ev.items():
    print(f'  - {nivel}: {count}/{total_rec} ({count/total_rec*100:.1f}%)')

print('\nDistribución por Dirección Clínica:')
for direccion, count in conteo_dir.items():
    print(f'  - {direccion}: {count}/{total_rec} ({count/total_rec*100:.1f}%)')

=== SÍNTESIS DE EVIDENCIA CANMAT (Total: 67 recomendaciones oficiales) ===
• Total con evidencia metodológica Level 1: 46/67 (68.7%)

Distribución de Niveles Metodológicos:
  - Level 1: 46/67 (68.7%)
  - Level 2: 18/67 (26.9%)
  - Level 4: 2/67 (3.0%)
  - Level 3: 1/67 (1.5%)

Distribución por Dirección Clínica:
  - Recomendado / Primera Línea: 38/67 (56.7%)
  - Recomendado / Segunda Línea: 18/67 (26.9%)
  - Recomendado / Tercera Línea: 5/67 (7.5%)
  - No Recomendado / Evidencia Negativa: 5/67 (7.5%)
  - No Recomendado / Alto Riesgo de Viraje: 1/67 (1.5%)


## 4. Conclusiones y Repercusiones Clínicas para el Motor RAG Sofia

1. **Jerarquización Racional y Separación de Nivel vs Dirección:** Las directrices CANMAT separan formalmente el **Nivel de Evidencia** metodológico (`Level 1` a `Level 4`) de la **Dirección de la Recomendación** (`Recomendado / Primera Línea`, `Recomendado / Segunda Línea`, `No Recomendado / Evidencia Negativa`, `No Recomendado / Alto Riesgo de Viraje`). Esta separación evita confundir la solidez metodológica de un ensayo clínico con su recomendación clínica favorable.
2. **Detección Determinista de Interacciones Enzimáticas CYP:** La matriz ampliada de 22 fármacos modela con precisión los bloqueos potentes de CYP2D6 (Paroxetina, Fluoxetina, Bupropión), la dependencia de CYP1A2 en Clozapina e inducción por tabaco, la vía de CYP2B6 para Bupropión, la inducción de CYP3A4 por Carbamazepina y la ausencia de metabolismo oxidativo CYP en Litio y Lamotrigina (glucuronidación UGT1A4).
3. **Advertencia de Alto Riesgo de Viraje Bipolar:** Los antidepresivos tricíclicos en monoterapia para depresión bipolar cuentan con evidencia de Nivel 1 en contra debido a su alto riesgo demostrado de inducir virajes maníacos y ciclación rápida, mientras que los ISRS se catalogan como Nivel 2 Negativo.
4. **Monitoreo de Toxicidad Grave y Paridad Relacional:** Los 505 efectos adversos de Jufe y 188 eventos de Stahl están 100% libres de negaciones erróneas y textos fragmentados, garantizando que el detector de cascadas iatrogénicas identifique intoxicaciones por litemia, hepatotoxicidad por valproato, agranulocitosis por clozapina y síndrome de Stevens-Johnson por lamotrigina con máxima sensibilidad clínica.